In [ ]:
import torch
import matplotlib.pyplot as plt

In [ ]:
from pfs.ga.bayesian import Model
from pfs.ga.bayesian.distributions import Normal
from pfs.ga.bayesian.proposals import NormalProposal
from pfs.ga.bayesian import MCMC
from pfs.ga.bayesian.kernels import GibbsKernel

In [ ]:
class Simple(Model):

    def __init__(self, N=(1000,)):
        super().__init__()

        self.N = N

    def model(self, context):
        N = self.N

        theta = context.sample('theta', Normal(0.0, 1.0))

        with context.plate('n', N):
            x = context.sample('x', Normal(theta, 1.0))
            obs = context.sample('obs', Normal(x, 0.01), observed=True)

    def step(self, context):
        context.step(
            'theta',
            [ self.theta ],
            proposal = NormalProposal(self.theta.value(context.state), 0.5)
        )

        context.step(
            'x',
            [ self.x ],
            proposal = NormalProposal(self.x.value(context.state), 1.0)
        )

In [ ]:
model = Simple()
model.build()

init_state = model.sample()
observed = { 'obs': init_state['obs'].clone() }

In [ ]:
# Print the hyperparameters
print('theta:', model.theta.value(init_state))

In [ ]:
# Plot the distribution of observed data
hist, bins = torch.histogram(observed['obs'], bins=30, density=True)
plt.step(bins[:-1], hist, where='post')

plt.axvline(model.theta.value(init_state), color='red', linestyle='--', label='True theta')

In [ ]:
kernel = GibbsKernel(model)
mcmc = MCMC(kernel,
            num_warmup=10000, num_samples=10000, num_chains=10,
            thinning=100,
            progress=True)

In [ ]:
mcmc.run(observed=observed)

In [ ]:
mcmc.trace['theta'].shape

In [ ]:
print('theta:', init_state['theta'])
print('theta:', torch.mean(mcmc.trace['theta'], dim=0))

In [ ]:
for i in range(mcmc.trace['theta'].shape[-1]):
    hist, bins = torch.histogram(mcmc.trace['theta'][:, i].flatten(), bins=30, density=True)
    plt.step(bins[:-1], hist, where='post')

plt.axvline(init_state['theta'], color='red', linestyle='--', label='True theta')

In [ ]:
for i in range(mcmc.trace['theta'].shape[-1]):
    plt.plot(mcmc.trace['theta'][..., i], '.')

plt.axhline(init_state['theta'], color='red', linestyle='--', label='True theta')

In [ ]:
mcmc.trace['x'].shape, observed['obs'].shape

In [ ]:
k = 0
for i in range(mcmc.trace['x'].shape[-1]):
    plt.plot(mcmc.trace['x'][:, k, i], '.')

plt.axhline(observed['obs'][k], color='red', linestyle='--', label='True theta')

In [ ]:
init_state['x'].shape, mcmc.trace['x'].shape

In [ ]:
init_state['x'][:10]

In [ ]:
torch.mean(mcmc.trace['x'], dim=(0, -1))[:10]